In [ ]:
# from torch import nn
import torch
# from torch.utils.data import Dataset, DataLoader
# from transformers.optimization import AdamW
# from tqdm import trange, tqdm_notebook

In [ ]:
from transformers.modeling_camembert import CamembertForQuestionAnswering
from transformers.tokenization_camembert import CamembertTokenizer

In [ ]:
camembert = CamembertForQuestionAnswering.from_pretrained('fmikaelian/camembert-base-fquad')
tokenizer = CamembertTokenizer.from_pretrained('fmikaelian/camembert-base-fquad'
                                               , do_lower_case=True
                                              )

In [ ]:
def split_text(context, length = 512,overlapp= 0):     # onlyText
    res = []
    tokens = tokenizer.encode(context, add_special_tokens=True)
    for i in range(0, int(len(tokens)-overlapp),(length-overlapp)):
            text = tokenizer.decode(tokens[i:i+length])
            text = text.replace('<s>','').replace('</s>','')
#             text = text.replace('</s>','')
#             print(text)
            res.append(text)
    return res
# print(split_text(text, overlapp=10))


# def split_text(question, context, overlapp= 0):      # Question and text
#     res = []
#     tokens = tokenizer.encode(question, context, add_special_tokens=True)
#     for i in range(0, int(len(tokens)-overlapp),(512-overlapp)):
#             question, text = tokenizer.decode(tokens[i:i+512])
# #             text = text.replace('<s>','')
# #             text = text.replace('</s>','')
            
#             print(text)
#             print("-------------------------")
# #             res.append(tokens[i:i+512])
#     return res



In [ ]:
# For each paragraph answer the question
def select_answer(question, text):
    # encode the question
    encoded_question = tokenizer.encode(question,add_special_tokens=True)
    length = len(encoded_question)
    encoded_text = split_text(context = text, length = 509-length)   # add overlapping    510
    max_score = 0
    answer = ""
    for split in encoded_text:
#         print(split)
        inp = tokenizer.encode(question, split, add_special_tokens=True)
        input_ids = torch.tensor([inp])
        with torch.no_grad():
            last_hidden_states = camembert(input_ids)[0]  # Models outputs are now tuples
            score = torch.max(last_hidden_states)
            if (max_score < score):
                # Update the answer
                max_score = score
                all_tokens = tokenizer.convert_ids_to_tokens(inp)
                start_scores, end_scores = camembert(input_ids)
                start = torch.argmax(start_scores)
                end = torch.argmax(end_scores)+1
                answer = "".join(all_tokens[start:end]).replace('▁',' ').strip()
#             print(answer)
    return answer

# Example: long Wikipedia article (Covid-19)

In [ ]:
text = """ La pandémie de Covid-19 est une pandémie d'une maladie infectieuse émergente, la maladie à coronavirus 2019 (Covid-19), provoquée par le coronavirus SARS-CoV-2. Elle apparaît le 17 novembre 2019 dans la ville de Wuhan, en Chine centrale, puis se propage dans le monde entier.

L'Organisation mondiale de la santé (OMS) alerte dans un premier temps la république populaire de Chine et ses autres États membres3, puis prononce l'état d'urgence de santé publique de portée internationale le 30 janvier3.

Le 13 janvier, un premier cas est découvert hors de Chine continentale. Deux navires de croisière (le MS Westerdam et le Diamond Princess) sont aussi touchés. Le nombre total de malades hors de la Chine dépasse les 1 500 à la mi-février1,4.

À partir de janvier 2020, le gouvernement chinois met en œuvre des procédures lourdes de confinement, et place plusieurs villes puis toute une région en quarantaine, fermant de nombreux sites publics et déployant d'importants moyens sanitaires.[réf. nécessaire]

Le 25 février, le nombre de nouveaux cas déclarés quotidiennement hors de Chine est plus élevé que dans ce pays5,6,7.

Le 11 mars 2020, l'épidémie de Covid-19 est déclarée pandémie8,9 par l'OMS, qui demande des mesures de protection essentielles pour prévenir la saturation des services de soins intensifs10 et renforcer l'hygiène préventive (suppression des contacts physiques, bises et serrements de mains, fin des attroupements et des grandes manifestations ainsi que des déplacements et voyages non indispensables, promotion du lavage des mains, mise en application de quarantaine, etc.). Cette pandémie mondiale provoque des annulations en série de manifestations sportives et culturelles sur toute la planète, la mise en place par de nombreux pays de mesures de confinement pour freiner la formation de nouveaux foyers de contagion, la fermeture des frontières de nombreux pays, et un krach boursier du fait des incertitudes et des craintes qu'elle fait peser sur l'économie mondiale. Elle a aussi des effets en termes d'instabilité sociale et économique et est le prétexte à la diffusion en ligne d'informations erronées ou relevant de la théorie du complot.[réf. nécessaire]

Le taux de létalité provisoire est plus bas que lors des précédentes épidémies à coronavirus11, avec pour estimation environ 1 à 2 % sur les cas dépistés12.

Au 8 avril, environ 1 529 500 cas cumulés sont confirmés dans le monde, dont environ 339 500 personnes guéries et 89 500 morts4. Ce sont 214 pays et territoires qui sont touchés par cette crise sanitaire mondiale, avec des foyers majeurs en Chine, en Italie, aux États-Unis, en Espagne, en Allemagne, en Iran et en France.

"""

questions = [
    "Quel est le taux de létalité ?",
    "Combien de cas confirmés dans le monde ?",
    "Quel est le nombre de nouveaux cas déclarés ?",
]

In [ ]:
for q in questions:
    print("La question est: "+q)
    print("La résponse est: "+select_answer(question = q,text = text))